In [1]:
!pip install transformers datasets sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.5 MB/s eta 0:00:00


In [3]:
!pip install datasets --upgrade

In [5]:
!pip install transformers datasets sentencepiece
!pip install evaluate

import torch
from transformers import MarianMTModel, MarianTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate # Import evaluate for metric function
import numpy as np

# Assuming you were using a metric like 'accuracy', you can now use:
metric = evaluate.load("accuracy")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.2 MB/s eta 0:00:00


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
# Specify the dataset and configuration
dataset = load_dataset("wmt14", "de-en")

# Inspect the dataset
print(dataset)


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 4508785
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3003
    })
})


In [7]:
# Select a smaller subset for training
small_train_dataset = dataset['train'].shuffle(seed=42).select([i for i in list(range(5000))])  # 5,000 samples
small_val_dataset = dataset['validation'].shuffle(seed=42).select([i for i in list(range(500))])  # 500 samples


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [8]:
SRC_LANG = "en"
TGT_LANG = "de"


In [11]:
model_name = f"Helsinki-NLP/opus-mt-{SRC_LANG}-{TGT_LANG}"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
model.to(device)


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(58101, 512, padding_idx=58100)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(58101, 512, padding_idx=58100)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [12]:
def preprocess_function(examples):
    inputs = [ex['translation'][SRC_LANG] for ex in examples['translation']]
    targets = [ex['translation'][TGT_LANG] for ex in examples['translation']]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [17]:
def preprocess_function(examples):
    # Assuming the dataset has a column named "translation" which contains a dictionary with keys 'en' and 'de'
    inputs = [ex[SRC_LANG] for ex in examples["translation"]]
    targets = [ex[TGT_LANG] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Before applying the map function, print the columns of the dataset to understand its structure
print(small_train_dataset.column_names)
# The output will help you identify the correct column names to use in preprocess_function
tokenized_train = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
tokenized_val = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])

['translation']


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [19]:
from transformers import TrainingArguments, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [23]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_steps=100,
    report_to="none"
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [24]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
)


In [25]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,1.132900,0.857270
2,0.907400,0.778819
3,0.835700,0.761497


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58100]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58100]], 'forced_eos_token_id': 0}


TrainOutput(global_step=939, training_loss=1.0431343582364572, metrics={'train_runtime': 341.3129, 'train_samples_per_second': 43.948, 'train_steps_per_second': 2.751, 'total_flos': 508475473920000.0, 'train_loss': 1.0431343582364572, 'epoch': 3.0})

In [28]:
trainer.save_model("fine-tuned-marianmt-en-de")
tokenizer.save_pretrained("fine-tuned-marianmt-en-de")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58100]], 'forced_eos_token_id': 0}


('fine-tuned-marianmt-en-de/tokenizer_config.json',
 'fine-tuned-marianmt-en-de/special_tokens_map.json',
 'fine-tuned-marianmt-en-de/vocab.json',
 'fine-tuned-marianmt-en-de/source.spm',
 'fine-tuned-marianmt-en-de/target.spm',
 'fine-tuned-marianmt-en-de/added_tokens.json')

In [29]:
from transformers import MarianMTModel, MarianTokenizer

# Load the fine-tuned model and tokenizer
model_path = "fine-tuned-marianmt-en-de"
tokenizer = MarianTokenizer.from_pretrained(model_path)
model = MarianMTModel.from_pretrained(model_path)
model.to(device)


/usr/local/lib/python3.10/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(58101, 512, padding_idx=58100)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(58101, 512, padding_idx=58100)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [30]:
def translate(text):
    # Tokenize the input text
    inputs = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt", max_length=128, truncation=True)
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # Generate translation using beam search
    with torch.no_grad():
        translated = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=128, num_beams=4, early_stopping=True)

    # Decode the translated tokens
    translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    return translated_text


In [31]:

import ipywidgets as widgets
from IPython.display import display


# Create input text area
input_text = widgets.Textarea(
    value='',
    placeholder='Enter text to translate',
    description='English:',
    layout=widgets.Layout(width='500px', height='100px')
)

# Create output text area
output_text = widgets.Textarea(
    value='',
    placeholder='Translation will appear here',
    description='German:',
    layout=widgets.Layout(width='500px', height='100px'),
    disabled=True
)

# Create a button to trigger translation
translate_button = widgets.Button(description='Translate')

def on_translate_button_clicked(b):
    input_string = input_text.value
    if input_string:
        translated_string = translate(input_string)
        output_text.value = translated_string
    else:
        output_text.value = "Please enter text to translate."

translate_button.on_click(on_translate_button_clicked)


# Display the widgets
display(input_text)
display(translate_button)
display(output_text)

Textarea(value='', description='English:', layout=Layout(height='100px', width='500px'), placeholder='Enter te…

Button(description='Translate', style=ButtonStyle())

Textarea(value='', description='German:', disabled=True, layout=Layout(height='100px', width='500px'), placeho…

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:4252: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:4252: FutureWarning: 
`prepare_seq2seq_batch` is